# Qwen3-4B medical fine-tune (training)

Trains a QLoRA adapter on ~17,700 examples from four medical sources -- sized
to measured T4 throughput, not the original 40,000-example estimate; see
"Build the training set" below. This notebook only trains. Evaluation
against two held-out benchmarks it never
saw -- MedQA-USMLE test (1,273) and MedMCQA validation (4,183) -- runs as a
**separate** Kaggle notebook once the evaluation code lands, fed the adapter
this notebook produces.

**Sidebar: Accelerator `GPU T4 x2`, Internet `On`.** Then Save Version ->
Save & Run All, rather than an interactive session that idles out.

The run goes straight to full scale with no calibration pass, so the probe at
step 50 aborts rather than letting a bad configuration burn six hours.


In [ ]:
# --- 1. Hardware check (stops here if the GPU is unusable) -----------------
import os, subprocess

# Two T4s are offered, but a 4B model in 4-bit is ~3.3GB against 15.6GB of card.
# Splitting it buys nothing and pays PCIe on every forward and backward. Pin to
# one GPU BEFORE torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

name = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                       "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
major, minor = torch.cuda.get_device_capability()
print(f"GPU         : {name}")
print(f"visible GPUs: {torch.cuda.device_count()} (pinned to one on purpose)")
print(f"capability  : {major}.{minor}")
print(f"torch       : {torch.__version__}   bf16: {torch.cuda.is_bf16_supported()}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. Compute capability {major}.{minor} ({name.split(',')[0]}) has no "
        "kernels in modern PyTorch builds.\nFIX: kernel-metadata.json pins "
        "machine_shape to a T4 already; if a P100 still arrived, set sidebar -> "
        "Accelerator -> 'GPU T4 x2' by hand, then Run All again.")
print("\nGPU supported." if torch.cuda.is_bf16_supported()
      else "\nGPU supported. Turing has no bf16; fp16 is selected automatically.")

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
# --- 2. Verify the install before spending GPU time on it ------------------
try:
    from unsloth import FastLanguageModel
    import trl, peft, transformers
    print(f"ok | transformers {transformers.__version__} | "
          f"trl {trl.__version__} | peft {peft.__version__}")
except Exception as e:
    print("INSTALL FAILED:", type(e).__name__, e)
    print("\nFallback, then Run > Restart session and skip the install cell:")
    print("  !pip install -q --upgrade --force-reinstall --no-cache-dir "
          "unsloth unsloth_zoo")

In [ ]:
# --- 3. Get the code from the attached dataset -----------------------------
import os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working/ft")
PKG   = WORK / "training"
PKG.mkdir(parents=True, exist_ok=True)

REQUIRED = {"records", "prompts", "sources", "prepare_data",
            "check_lengths", "budget", "train"}

EVAL_REQUIRED = {"records", "prompts", "sources", "evalcore", "evaluate",
                 "modeling"}

RUN2_REQUIRED = {"records", "prompts", "sources", "sources_v2", "format_v2",
                 "evalcore", "evaluate", "modeling", "train", "budget",
                 "eval_worker", "eval_report", "kaggle_paths"}

DATA_FILES = ("train.jsonl", "eval_medqa.jsonl", "eval_medmcqa.jsonl",
              "eval_pubmedqa.jsonl", "eval_mmlu_medical.jsonl", "data_report.json")


def find_code_dir(root: Path, required: set[str] = REQUIRED) -> Path:
    """Locate the uploaded modules wherever Kaggle mounted them.

    The mount path is not stable: a dataset declared as gb1105/medical-ft-code
    turned up under /kaggle/input/datasets/... rather than at
    /kaggle/input/medical-ft-code. Two runs died on that assumption, so this
    searches for the directory that actually holds the modules instead.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-code.")
    candidates = []
    for path in root.rglob("*.py"):
        stems = {p.stem for p in path.parent.glob("*.py")}
        if required <= stems:
            candidates.append(path.parent)
    if not candidates:
        found = sorted(str(p.relative_to(root)) for p in root.rglob("*.py"))[:20]
        tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
        raise SystemExit(
            f"\nSTOP. No directory under {root} contains all of {sorted(required)}.\n"
            f".py files found: {found or 'none'}\n"
            f"First entries under /kaggle/input: {tree}\n"
            "FIX: re-run scripts/push_kaggle.sh, then confirm the "
            "medical-ft-code dataset is attached in the sidebar.")
    return sorted(set(candidates))[0]


def find_adapter_dir(root: Path) -> Path:
    """Locate the uploaded LoRA adapter by content, the same way.

    A directory qualifies when adapter_config.json and adapter_model.safetensors
    sit side by side. A final adapter wins over any checkpoint-* directory, so a
    stray checkpoint cannot be scored in place of the finished run.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-adapter.")
    found = sorted({p.parent for p in root.rglob("adapter_config.json")
                    if (p.parent / "adapter_model.safetensors").exists()})
    final = [d for d in found if not d.name.startswith("checkpoint-")]
    if final or found:
        return (final or found)[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No LoRA adapter under {root}: no adapter_config.json with "
        "adapter_model.safetensors beside it.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_adapter.sh, then attach medical-ft-adapter in "
        "the sidebar.")


def code_fingerprint(directory: Path) -> str:
    """A short hash of every .py file's name and bytes, in name order.

    Each notebook is built for one exact set of modules. If Kaggle mounts an
    older version of the code dataset -- a new version still processing, or an
    old one attached by hand -- the modules carry the right names and the wrong
    code, and nothing else would notice.
    """
    import hashlib

    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode() + b"\0" + path.read_bytes() + b"\0")
    return digest.hexdigest()[:16]


def find_data_dir(root: Path) -> Path:
    """Locate the uploaded run 2 data by content, like find_code_dir."""
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-data.")
    found = sorted({p.parent for p in root.rglob("train.jsonl")
                    if all((p.parent / name).exists() for name in DATA_FILES)})
    if found:
        return found[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No directory under {root} holds all of {list(DATA_FILES)}.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_data.sh, then attach medical-ft-data in the sidebar.")


def data_fingerprint(directory: Path) -> str:
    """The same idea as code_fingerprint, for the frozen data files."""
    import hashlib

    digest = hashlib.sha256()
    for name in DATA_FILES:
        digest.update(name.encode() + b"\0" + (directory / name).read_bytes() + b"\0")
    return digest.hexdigest()[:16]

SRC = find_code_dir(INPUT)
EXPECTED_FINGERPRINT = "b3f254b10286ade0"
if code_fingerprint(SRC) != EXPECTED_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached code ({code_fingerprint(SRC)}) is not the code this "
        f"notebook was built for ({EXPECTED_FINGERPRINT}).\n"
        "Kaggle may still be processing a new version of medical-ft-code, or an "
        "older version is attached.\n"
        "FIX: wait a minute and re-run; if it persists, run scripts/push_kaggle.sh again.")
print("found the code at:", SRC)

for src_file in sorted(SRC.glob("*.py")):
    shutil.copy(src_file, PKG / src_file.name)
(PKG / "__init__.py").touch()

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd(), "|", len(list(PKG.glob("*.py"))), "modules:",
      ", ".join(sorted(p.stem for p in PKG.glob("*.py"))))

# A failing `!python x.py` returns non-zero but does not raise in Jupyter, so the
# notebook would sail past a dead step and fail later somewhere confusing.
def step(cmd: str):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed (exit {p.returncode}):\n  {cmd}")
    print("\nok\n", flush=True)

## 4. Build the training set

6,300 MedMCQA + 3,700 MedQA + 2,600 medical-o1 + 5,400 ChatDoctor -- 18,000
requested, holding the spec's 70% exam/reasoning and 30% patient-dialogue
split (12,600 / 5,400). This size is set by measured throughput, not chosen
in advance: the step-50 probe on run 1 measured 35.5s/step at effective
batch 32 on a single T4, i.e. 0.9 examples/sec, and 40,000 examples at that
rate is a 12-hour run against a 9-hour Kaggle session cap. 18,000 lands
around 17,700 after decontamination and the length drop, which is 552 steps
-- about 5.4 hours, inside the 6-hour training budget with real margin.
Both benchmarks load first so every training row can be decontaminated
against them.

In [ ]:
step("python -m training.prepare_data --medmcqa 6300 --medqa 3700 "
     "--medical-o1 2600 --chatdoctor 5400 --val-size 500 --out data")

import json
print(json.dumps(json.load(open("data/report.json")), indent=2))

## 5. Measure sequence length

Vignettes plus chain-of-thought run far longer than a guessed default. `max_seq`
comes from the measured p99, and the outliers above it are dropped rather than
truncated mid-answer.

In [ ]:
import re, subprocess

out = subprocess.run("python -m training.check_lengths --data data/train.jsonl",
                     shell=True, capture_output=True, text=True).stdout
print(out)
MAX_SEQ = int(re.search(r"suggested --max-seq (\d+)", out).group(1))
print("MAX_SEQ =", MAX_SEQ)

step(f"python -m training.check_lengths --data data/train.jsonl "
     f"--max-seq {MAX_SEQ} --drop")
step(f"python -m training.check_lengths --data data/val.jsonl "
     f"--max-seq {MAX_SEQ} --drop")

## 6. Train

The probe at step 50 extrapolates and aborts if the run will not fit six hours.
If it stops here, halve `--batch-size` and double `--grad-accum` as instructed
and re-run: the effective batch is unchanged.

In [ ]:
step(f"python -m training.train --data data --out outputs/run1 "
     f"--max-seq {MAX_SEQ} --batch-size 8 --grad-accum 4 --rank 32 "
     f"--epochs 1 --save-steps 200 --budget-seconds 21600")

In [ ]:
# --- 7. Training stats, and save everything to the Output panel ------------
import json, shutil
from pathlib import Path

stats = json.loads(Path("outputs/run1/train_stats.json").read_text())
print("\n########## Training stats ##########")
for key, value in stats.items():
    print(f"  {key}: {value}")

out = Path("/kaggle/working")
shutil.copy("data/report.json", out / "data_report.json")
shutil.make_archive(str(out / "run1-adapter"), "zip", "outputs/run1")
print("\nSaved to /kaggle/working -- download run1-adapter.zip from the "
      "Output panel.")

## Done

`run1-adapter.zip` and `data_report.json` are in the **Output** panel. The
adapter was trained on ~17,700 examples (not the spec's original 40,000 --
see "Build the training set" above for why). Download `run1-adapter.zip`
and feed it to the evaluation notebook, which runs as a separate Kaggle
session once `training/evalcore.py` and `training/evaluate.py` exist.